# Задача 2. PPO и Discrete SAC на четырёх средах GridWorld

Сравнения:
- **PPO + GRU**, **PPO + MLP** (без памяти);
- **Discrete SAC + GRU**, **Discrete SAC + MLP**;
- Discrete SAC с `soft_q=True` против `soft_q=False` (на Env2 и Env4).

Все прогоны — один сид (`seed=0`). Результаты пишутся в `outputs/task2/`:
`runs/<имя>/` (config, metrics.csv по эпизодам, model.pt, eval.json, status.json, train.log), `figures/`, `eval/`, `tables/`.
Графики строятся по сохранённым CSV, поэтому их можно перерисовать без обучения: `RUN_STAGES=plots`.

In [1]:
import json
import os
import sys
from pathlib import Path

# работаем из корня репозитория: там лежат src/ и outputs/
if Path.cwd().name == "tasks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import torch

from src import utils
from src.envs import make_env, optimal_stats
from src.models import DSAC, PPO

# Режим запуска. При обычном открытии ноутбука: полное обучение всех экспериментов и все графики.
# На сервере скрипт задаёт переменные окружения, чтобы запускать эксперименты параллельно:
#   RUN_MODE=smoke|full, RUN_ONLY=<имя эксперимента> (пусто — все), RUN_STAGES=train,plots
RUN_MODE = os.environ.get("RUN_MODE", "full")
RUN_ONLY = os.environ.get("RUN_ONLY", "")
RUN_STAGES = os.environ.get("RUN_STAGES", "train,plots").split(",")
torch.set_num_threads(1)  # много прогонов идут параллельно, каждому хватит одного потока CPU
CUDA = torch.cuda.is_available()
print("режим:", RUN_MODE, "| эксперимент:", RUN_ONLY or "все", "| этапы:", RUN_STAGES, "| CUDA:", CUDA)

SETUPS = {
    1: dict(h=5, w=5, n_colors=None, obstacle_fraction=0.0),  # 25 уникальных цветов — MDP
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}

режим: full | эксперимент: dsac_env2_gru_hardq_seed0 | этапы: ['train'] | CUDA: True


In [2]:
# Базовые гиперпараметры (те же, что в первой версии ноутбуков). Для каждого эксперимента делаем копию и меняем нужное.
ppo_cfg = {
    "device": "cpu",  # PPO на GPU не быстрее (замер на сервере: 479 против 429 шагов/с)
    "env_seed": 0,  # от него зависит карта
    "seed": 0,  # от него зависят сеть и старты эпизодов
    "memory": True,  # True — GRU, False — MLP без памяти
    "episodes_per_update": 10,  # сколько эпизодов собрать перед обновлением
    "epochs": 4,
    "episodes_per_batch": 5,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "lr": 3e-4,
    "clip": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "hidden": 128,
    "save_every": 50,  # как часто сохранять метрики и веса на диск
}
dsac_cfg = {
    "device": "cuda" if CUDA else "cpu",  # DSAC на GPU в ~3 раза быстрее (167 против 52 шагов/с)
    "env_seed": 0,
    "seed": 0,
    "memory": True,
    "soft_q": True,  # мягкая Q (энтропия в цели критика) или обычная
    "buffer_episodes": 5000,  # сколько последних эпизодов помнить
    "learning_starts": 20,  # сколько эпизодов только копим опыт, не учась
    "updates_per_episode": 4,  # сколько обновлений после каждого эпизода
    "batch_size": 8,  # сколько случайных эпизодов из буфера в одном обновлении
    "gamma": 0.99,
    "tau": 0.005,  # скорость, с которой target-критики догоняют обычных
    "lr": 3e-4,
    "alpha": 0.005,  # температура: вес энтропии
    "hidden": 128,
    "save_every": 25,
}
# число эпизодов обучения: на больших картах дольше. В режиме smoke — коротко, только проверить, что всё работает
EPISODES = {
    "full": {"ppo": {1: 1000, 2: 1000, 3: 1500, 4: 1500}, "dsac": {1: 500, 2: 500, 3: 800, 4: 800}},
    "smoke": {"ppo": {1: 20, 2: 20, 3: 20, 4: 20}, "dsac": {1: 8, 2: 8, 3: 4, 4: 4}},
}
EVAL_EPISODES = {"full": 100, "smoke": 5}

# smoke-прогоны пишем отдельно, чтобы не смешать их с полными
OUT = "outputs/task2" if RUN_MODE == "full" else "runs/smoke/task2"

## Список экспериментов

In [3]:
EXPERIMENTS = []
for i in SETUPS:
    for memory in [True, False]:
        tag = "gru" if memory else "mlp"
        EXPERIMENTS.append({"name": f"ppo_env{i}_{tag}_seed0", "algo": "ppo", "cfg": {
            **ppo_cfg, **SETUPS[i], "memory": memory, "episodes": EPISODES[RUN_MODE]["ppo"][i],
            "budget": RUN_MODE, "out_dir": f"{OUT}/runs", "run_name": f"ppo_env{i}_{tag}_seed0"}})
        EXPERIMENTS.append({"name": f"dsac_env{i}_{tag}_seed0", "algo": "dsac", "cfg": {
            **dsac_cfg, **SETUPS[i], "memory": memory, "episodes": EPISODES[RUN_MODE]["dsac"][i],
            "budget": RUN_MODE, "out_dir": f"{OUT}/runs", "run_name": f"dsac_env{i}_{tag}_seed0"}})
# проверка требования задания: DSAC с обычной (не мягкой) Q — на двух показательных средах
for i in [2, 4]:
    EXPERIMENTS.append({"name": f"dsac_env{i}_gru_hardq_seed0", "algo": "dsac", "cfg": {
        **dsac_cfg, **SETUPS[i], "soft_q": False, "episodes": EPISODES[RUN_MODE]["dsac"][i],
        "budget": RUN_MODE, "out_dir": f"{OUT}/runs", "run_name": f"dsac_env{i}_gru_hardq_seed0"}})

for exp in EXPERIMENTS:
    print(exp["name"], "| эпизодов:", exp["cfg"]["episodes"], "|", exp["cfg"]["device"])

ppo_env1_gru_seed0 | эпизодов: 1000 | cpu
dsac_env1_gru_seed0 | эпизодов: 500 | cuda
ppo_env1_mlp_seed0 | эпизодов: 1000 | cpu
dsac_env1_mlp_seed0 | эпизодов: 500 | cuda
ppo_env2_gru_seed0 | эпизодов: 1000 | cpu
dsac_env2_gru_seed0 | эпизодов: 500 | cuda
ppo_env2_mlp_seed0 | эпизодов: 1000 | cpu
dsac_env2_mlp_seed0 | эпизодов: 500 | cuda
ppo_env3_gru_seed0 | эпизодов: 1500 | cpu
dsac_env3_gru_seed0 | эпизодов: 800 | cuda
ppo_env3_mlp_seed0 | эпизодов: 1500 | cpu
dsac_env3_mlp_seed0 | эпизодов: 800 | cuda
ppo_env4_gru_seed0 | эпизодов: 1500 | cpu
dsac_env4_gru_seed0 | эпизодов: 800 | cuda
ppo_env4_mlp_seed0 | эпизодов: 1500 | cpu
dsac_env4_mlp_seed0 | эпизодов: 800 | cuda
dsac_env2_gru_hardq_seed0 | эпизодов: 500 | cuda
dsac_env4_gru_hardq_seed0 | эпизодов: 800 | cuda


## Обучение и оценка

Уже завершённые прогоны (`status.json` = completed) пропускаются, поэтому ячейку можно перезапускать.

In [4]:
def run_experiment(exp):
    """Обучает модель, оценивает её и сохраняет всё в OUT/runs/<имя>/ и OUT/eval/."""
    run_dir = Path(OUT) / "runs" / exp["name"]
    status_file = run_dir / "status.json"
    if status_file.exists() and json.loads(status_file.read_text())["status"] == "completed":
        print("уже готово, пропускаю:", exp["name"])
        return
    print("=" * 20, exp["name"], "=" * 20)
    if exp["algo"] == "ppo":
        alg = PPO(exp["cfg"])
    else:
        alg = DSAC(exp["cfg"])
    alg.train()

    # оценка на одних и тех же стартах (сиды 10000, 10001, ...): жадно и с сэмплированием действий
    n = EVAL_EPISODES[RUN_MODE]
    greedy = alg.evaluate(n, greedy=True)
    sampled = alg.evaluate(n, greedy=False)
    results = {"budget": RUN_MODE,
               "greedy": utils.eval_summary(greedy, alg.cfg["gamma"], alg.optimal),
               "stochastic": utils.eval_summary(sampled, alg.cfg["gamma"], alg.optimal)}
    utils.save_eval(alg.run_dir, results)
    print("оценка:", json.dumps(results, ensure_ascii=False))

    # тепловая карта и GIF — по эпизодам с сэмплированием (так агент действует при обучении)
    eval_dir = Path(OUT) / "eval"
    eval_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 6))
    image = utils.plot_heatmap(ax, alg.env, sampled)
    fig.colorbar(image, ax=ax, fraction=0.046, label="посещений за эпизод")
    s = results["stochastic"]
    ax.set_title(f"{exp['name']}: success {s['success_rate']:.2f}, длина {s['mean_length']:.1f} "
                 f"(оптимум {s['optimal_length']:.1f})", fontsize=9)
    fig.savefig(eval_dir / f"{exp['name']}_heatmap.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    short = [ep for ep in sampled if ep["length"] <= 60][:5]
    if len(short) == 0:
        short = sampled[:1]
    utils.save_gif(alg.env, short, eval_dir / f"{exp['name']}.gif", exp["name"])


if "train" in RUN_STAGES:
    for exp in EXPERIMENTS:
        if RUN_ONLY == "" or RUN_ONLY == exp["name"]:
            run_experiment(exp)

==================== dsac_env2_gru_hardq_seed0 ====================


/root/smallGridWorld-experiments/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1440: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1481.)
  result = _VF.gru(


episode 50 | steps 1537 | return 0.730 (opt 0.975) | length 30.7 | entropy 1.05


episode 100 | steps 2750 | return 0.776 (opt 0.975) | length 24.3 | entropy 0.39


episode 150 | steps 3163 | return 0.931 (opt 0.975) | length 8.3 | entropy 0.25


...

episode 400 | steps 4481 | return 0.963 (opt 0.975) | length 4.8 | entropy 0.15


episode 450 | steps 4674 | return 0.972 (opt 0.975) | length 3.9 | entropy 0.13


episode 500 | steps 4908 | return 0.964 (opt 0.975) | length 4.7 | entropy 0.12


оценка: {"budget": "full", "greedy": {"episodes": 100, "success_rate": 1.0, "failures": 0, "mean_return": 0.9677849015117413, "mean_length": 4.28, "mean_success_length": 4.28, "optimal_length": 3.5416666666666665, "optimal_return": 0.9749099703833749}, "stochastic": {"episodes": 100, "success_rate": 1.0, "failures": 0, "mean_return": 0.9662283849987614, "mean_length": 4.44, "mean_success_length": 4.44, "optimal_length": 3.5416666666666665, "optimal_return": 0.9749099703833749}}


saved outputs/task2/eval/dsac_env2_gru_hardq_seed0.gif


## Графики (по сохранённым CSV)

In [5]:
plt.style.use("seaborn-v0_8-darkgrid")
COLORS = ["tab:red", "tab:blue", "tab:green", "tab:orange", "tab:purple"]


def load_metrics(name):
    """metrics.csv прогона как словарь {колонка: np.array}; None, если прогона нет."""
    path = Path(OUT) / "runs" / name / "metrics.csv"
    if not path.exists():
        print("нет данных:", name)
        return None
    import csv
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    data = {}
    for key in rows[0]:
        values = []
        for row in rows:
            if row[key] == "True":
                values.append(1.0)
            elif row[key] == "False":
                values.append(0.0)
            elif row[key] in ("", "nan"):
                values.append(np.nan)
            else:
                values.append(float(row[key]))
        data[key] = np.array(values)
    return data


def plot_curve(ax, values, label, color, window=50):
    """Линия — скользящее среднее за window эпизодов, полоса — скользящее стандартное отклонение
    внутри ОДНОГО прогона (1 сид). Это не доверительный интервал по сидам."""
    means, stds = [], []
    for i in range(len(values)):
        last = values[max(0, i - window + 1):i + 1]
        last = last[~np.isnan(last)]
        means.append(np.mean(last) if len(last) else np.nan)
        stds.append(np.std(last) if len(last) else np.nan)
    means, stds = np.array(means), np.array(stds)
    x = np.arange(len(values))
    ax.plot(x, means, color=color, linewidth=2, label=label)
    ax.fill_between(x, means - stds, means + stds, color=color, alpha=0.25)


def plot_metric_grid(names, labels, key, ylabel, title, filename, optimal_key=None):
    """По графику на каждую среду; на графике — несколько прогонов (names[i] — список имён для Env i+1)."""
    fig, axes = plt.subplots(1, len(names), figsize=(5.5 * len(names), 4.3), squeeze=False)
    for ax, (env_name, run_names) in zip(axes[0], names.items()):
        for j, (run, label) in enumerate(zip(run_names, labels)):
            data = load_metrics(run)
            if data is None:
                continue
            plot_curve(ax, data[key], label, COLORS[j])
            if optimal_key is not None and j == 0:
                config = json.loads((Path(OUT) / "runs" / run / "config.json").read_text())
                ax.axhline(config["optimal"][optimal_key], color="black", linestyle=":", label="оптимум (oracle)")
        ax.set_title(f"{env_name}")
        ax.set_xlabel("эпизод")
        ax.set_ylabel(ylabel)
        if key == "length":
            ax.set_yscale("log")
        ax.legend(fontsize=8)
    fig.suptitle(f"{title} (линия — скользящее среднее за 50 эпизодов, полоса — ± ст. откл. в окне; 1 сид)")
    fig.tight_layout()
    path = Path(OUT) / "figures" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


def eval_table(names):
    """Таблица итоговой оценки по сохранённым eval.json."""
    rows = []
    for name in names:
        path = Path(OUT) / "runs" / name / "eval.json"
        if not path.exists():
            rows.append({"run": name, "status": "pending"})
            continue
        result = json.loads(path.read_text(encoding="utf-8"))
        for mode in ["greedy", "stochastic"]:
            r = result[mode]
            rows.append({"run": name, "budget": result["budget"], "eval": mode,
                         "success_rate": round(r["success_rate"], 3), "mean_return": round(r["mean_return"], 3),
                         "mean_length": round(r["mean_length"], 1),
                         "mean_success_length": None if r["mean_success_length"] is None else round(r["mean_success_length"], 1),
                         "failures": r["failures"], "optimal_length": round(r["optimal_length"], 2)})
    return rows

In [6]:
if "plots" in RUN_STAGES:
    for algo in ["ppo", "dsac"]:
        names = {f"Env{i}": [f"{algo}_env{i}_gru_seed0", f"{algo}_env{i}_mlp_seed0"] for i in SETUPS}
        labels = ["GRU (с памятью)", "MLP (без памяти)"]
        A = algo.upper()
        plot_metric_grid(names, labels, "return", "средняя дисконтированная отдача", f"{A}: отдача", f"{algo}_return.png", "return")
        plot_metric_grid(names, labels, "success", "доля успешных эпизодов", f"{A}: успех", f"{algo}_success.png")
        plot_metric_grid(names, labels, "length", "длина эпизода", f"{A}: длина эпизода", f"{algo}_length.png", "length")
        plot_metric_grid(names, labels, "loss", "полный лосс", f"{A}: полный лосс", f"{algo}_loss.png")
        if algo == "ppo":
            plot_metric_grid(names, labels, "policy_loss", "лосс актора", "PPO: лосс актора (policy)", "ppo_policy_loss.png")
            plot_metric_grid(names, labels, "value_loss", "лосс критика", "PPO: лосс критика (value)", "ppo_value_loss.png")
        else:
            plot_metric_grid(names, labels, "q_loss", "лосс критиков", "DSAC: лосс критиков", "dsac_q_loss.png")
            plot_metric_grid(names, labels, "actor_loss", "лосс актора", "DSAC: лосс актора", "dsac_actor_loss.png")
        plot_metric_grid(names, labels, "entropy", "энтропия стратегии", f"{A}: энтропия", f"{algo}_entropy.png")
        plot_metric_grid(names, labels, "explore_prob", "вероятность нежадного действия", f"{A}: исследование", f"{algo}_explore_prob.png")

In [7]:
if "plots" in RUN_STAGES:
    # PPO против DSAC (оба с памятью)
    names = {f"Env{i}": [f"ppo_env{i}_gru_seed0", f"dsac_env{i}_gru_seed0"] for i in SETUPS}
    plot_metric_grid(names, ["PPO + GRU", "DSAC + GRU"], "return", "средняя дисконтированная отдача",
                     "PPO против DSAC", "ppo_vs_dsac_return.png", "return")
    # мягкая Q против обычной
    names = {f"Env{i}": [f"dsac_env{i}_gru_seed0", f"dsac_env{i}_gru_hardq_seed0"] for i in [2, 4]}
    plot_metric_grid(names, ["soft_q=True", "soft_q=False"], "return", "средняя дисконтированная отдача",
                     "DSAC: мягкая Q против обычной", "dsac_soft_q_return.png", "return")

## Итоговая оценка обученных моделей

100 eval-эпизодов на одних и тех же стартах (сиды 10000…10099), жадно и с сэмплированием действий. `optimal_length` — средняя длина кратчайшего пути у oracle с доступом к координатам.

In [8]:
if "plots" in RUN_STAGES:
    rows = eval_table([exp["name"] for exp in EXPERIMENTS])
    table_dir = Path(OUT) / "tables"
    table_dir.mkdir(parents=True, exist_ok=True)
    keys = sorted({k for row in rows for k in row}, key=lambda k: list(rows[0]).index(k) if k in rows[0] else 99)
    import csv
    with open(table_dir / "eval_summary.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["run", "status", "budget", "eval", "success_rate", "mean_return",
                                               "mean_length", "mean_success_length", "failures", "optimal_length"])
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)